# 06_Dataset与批次

**学习目标：** 实现__len__和__getitem__，用DataLoader组成batch，保持划分独立。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

Dataset回答“第i个样本是什么”，DataLoader负责组batch、打乱顺序等。训练集和验证集使用不同清单。本机入门统一用num_workers=0，避免多进程让报错难以定位。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 一个最小真实片段Dataset


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
torch.set_num_threads(4)
class ClipDataset(Dataset):
    def __init__(self, split='train', limit=4):
        self.split = split
        self.rows = L.manifest(split)['samples'][:limit]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        clip, label, _ = L.load_clip(self.split, index, frames=4, size=32)
        return torch.from_numpy(clip), label
dataset = ClipDataset()
clip, label = dataset[0]
print(len(dataset), clip.shape, label)


## 由单样本组成batch


In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
x, y = next(iter(loader))
print('输入', x.shape, '标签', y.shape, y.dtype)
assert x.shape == (2, 4, 4, 32, 32)
print('本课小输入用于读懂批次，不是比赛模型的正式输入配置。')


## 划分不是把列表随便拼起来


In [ ]:
validation = ClipDataset('validation')
print('train路径', L.sample_path('train', 0)[0])
print('validation路径', L.sample_path('validation', 0)[0])
print('训练只迭代train；validation只评估。评估时通常不shuffle、不做随机增强。')


## 动手练习

把batch_size改为3。四个样本且drop_last=False时应产生几个batch、大小各是多少？填入列表your_answer。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：[第一个batch大小, 第二个batch大小]
if your_answer is None:
    print('待完成：最后不足3个仍保留')
else:
    assert your_answer == [3, 1]
    print('练习通过')


## 常见误区

input的batch轴与类别轴不同；最后一个batch可能更小，不把batch大小硬编码到模型。


## 过关检查与选读

解释Dataset/DataLoader分工；指出B/C/T/H/W；说清num_workers=0用途。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/06_Dataset与批次.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
